In [1]:
import os
import sys
import json
import glob
import gc
import hashlib

import numpy as np
import pandas as pd

sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils            
import Snowflake_configuration

from snowflake.snowpark.session import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType

session = Session.builder.configs(Snowflake_configuration.snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

### Read the table

In [2]:
df = session.sql("""SELECT * 
                 FROM MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES
                 WHERE PARENT_DEALER_CODE_MODEL_FAMILY IN 
                 (SELECT PARENT_DEALER_CODE_MODEL_FAMILY
                  FROM MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING)""")

snowflake_utils.shape_of_snowpark_df(df)

(43529400, 71)

In [3]:
#Confirming that only the series A is selected
df.select(F.count_distinct("PARENT_DEALER_CODE_MODEL_FAMILY")).show()

------------------------------------------------------
|"COUNT( DISTINCT ""PARENT_DEALER_CODE_MODEL_FAM...  |
------------------------------------------------------
|32244                                               |
------------------------------------------------------



In [4]:
#Removing columns
df = df.drop("MODEL_NAME","MODEL_FAMILY_CODE","FIRST_SALE_DATE")

In [5]:
df = df.with_column("DEALER_CITY", F.initcap(F.trim(F.col("DEALER_CITY"))))

In [6]:
df.select(F.col("DEALER_CITY")).distinct().show(df.select(F.count_distinct("DEALER_CITY")).collect()[0][0])

------------------------------
|"DEALER_CITY"               |
------------------------------
|Theni                       |
|Chhindwara                  |
|Gondiya                     |
|Gangoh                      |
|Karimganj                   |
|Betul                       |
|Shrirampur                  |
|Bahadurgarh                 |
|Dhanbad                     |
|Jhalawar                    |
|Nayagarh                    |
|Vapi                        |
|Shyamsundar                 |
|Chittoor                    |
|Sirsa                       |
|Valsad                      |
|Gandhinagar                 |
|Surendranagar               |
|Patancheru                  |
|Namakkal                    |
|Sitalpur                    |
|Gaura Chowki                |
|Ambala                      |
|Kampur                      |
|Bhimnagar                   |
|Bareilly                    |
|Pali                        |
|Amravati                    |
|Panipat                     |
|Simariy

In [16]:
local_train_dir = "./local_train_data"
calendar_path   = "./shared_calendar.parquet"
vocab_path      = "./category_vocab.json"
roles_path      = "./column_roles.json"
TABLE_NAME        = "MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES"
VALID_SERIES_TBL  = "MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING"

time_col   = "CAL_DATE"
group_col  = "PARENT_DEALER_CODE_MODEL_FAMILY"
dealer_col = "PARENT_DEALER_CODE"
target_col = "NET_SALES"
FREQ       = "D"
CHUNK_SIZE = 100                      # dealers per parquet file

TRAIN_START    = "2023-04-01"
TRAIN_END      = "2026-04-30"
VAL_START      = "2025-05-01"
VAL_END        = "2026-08-31"
FORECAST_START = "2026-09-01"
FORECAST_END   = "2026-12-10"

INPUT_CHUNK_LENGTH  = 365
OUTPUT_CHUNK_LENGTH = (pd.Timestamp(FORECAST_END) - pd.Timestamp(FORECAST_START)).days + 1  

In [8]:
# columns that are strings but are NOT static covariates
NON_STATIC_STRING_COLS = ["DAY_OF_THE_WEEK", group_col]
# raw calendar columns: replaced by hand-built features in Section 6
CALENDAR_RAW_COLS = ["YEAR", "MONTH", "DAY_OF_THE_MONTH", "DAY_OF_THE_WEEK"]


# --- sanity checks on the configuration itself --------------------------------
_d = {k: pd.Timestamp(v) for k, v in dict(TRAIN_START=TRAIN_START, TRAIN_END=TRAIN_END,
                                           VAL_START=VAL_START, VAL_END=VAL_END,
                                           FORECAST_START=FORECAST_START, FORECAST_END=FORECAST_END).items()}
assert _d["TRAIN_START"] < _d["TRAIN_END"] < _d["VAL_END"] < _d["FORECAST_START"] <= _d["FORECAST_END"]
assert _d["FORECAST_START"] == _d["VAL_END"] + pd.Timedelta(days=1), \
    "History must end the day before the forecast starts."
_val_days = (_d["VAL_END"] - _d["VAL_START"]).days + 1
assert _val_days >= INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH, \
    f"VAL period is {_val_days} days; one validation window needs {INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH}."
_val_scored_start = _d["VAL_START"] + pd.Timedelta(days=INPUT_CHUNK_LENGTH)
print(f"ICL / OCL               : {INPUT_CHUNK_LENGTH} / {OUTPUT_CHUNK_LENGTH}")
print(f"Validation windows      : {_val_days - INPUT_CHUNK_LENGTH - OUTPUT_CHUNK_LENGTH + 1} per series")
print(f"Validation scored dates : {_val_scored_start.date()} -> {_d['VAL_END'].date()}")
print(f"Overlap with training targets: "
      f"{'NONE' if _val_scored_start > _d['TRAIN_END'] else 'YES -- val_loss partly measures memorisation'}")


ICL / OCL               : 365 / 101
Validation windows      : 23 per series
Validation scored dates : 2026-05-01 -> 2026-08-31
Overlap with training targets: NONE


In [9]:
# strip quote characters Snowflake keeps in some column names
for old in df.columns:
    new = old.replace('"', "")
    if new != old:
        df = df.rename(old, new)

df = df.with_column(group_col, F.replace(F.col(group_col), F.lit("<>"), F.lit("_")))


In [10]:
id_cols = [time_col, group_col, target_col]

static_covariates = [
    f.name.replace('"', "") for f in df.schema
    if isinstance(f.datatype, StringType)
    and f.name.replace('"', "") not in NON_STATIC_STRING_COLS
]
all_cols = [c.replace('"', "") for c in df.columns]      # Snowpark quotes names like "N-16"
festive_covariates = [
    c for c in all_cols
    if c not in static_covariates and c not in id_cols
    and c not in CALENDAR_RAW_COLS and c not in NON_STATIC_STRING_COLS
]

assert "MODEL_FAMILY" in static_covariates, "MODEL_FAMILY must be a static covariate."
assert "MODEL_NAME" not in static_covariates, "MODEL_NAME is 1:1 with MODEL_FAMILY and should have been dropped."
assert dealer_col in static_covariates

print("static covariates :", static_covariates)
print(f"festive covariates: {len(festive_covariates)} ->", festive_covariates)

static covariates : ['PARENT_DEALER_CODE', 'MODEL_FAMILY', 'BRAKE_TYPE', 'IGNITION_TYPE', 'WHEEL_TYPE', 'COLOUR', 'X_CITY_CATEGORY', 'ZONAL_OFFICE_NAME', 'DEALER_CITY']
festive covariates: 52 -> ['HARTALIK_TEEJ', 'GANESH_CHATURTHI', 'JANMASHTAMI', 'VISHWAKARMA_PUJA', 'KARWA_CHAUTH', 'ONAM', 'N-16', 'N-15', 'N-14', 'N-13', 'N-12', 'N-11', 'N-10', 'N-9', 'N-8', 'N-7', 'N-6', 'N-5', 'N-4', 'N-3', 'N-2', 'N-1', 'N', 'N+1', 'N+2', 'N+3', 'N+4', 'N+5', 'N+6', 'N+7', 'N+8', 'N+9', 'N+10', 'D-3', 'D-2', 'D-1', 'D', 'D+1', 'D+2', 'D+3', 'D+4', 'D+5', 'D+6', 'HANUMAN_JAYANTI', 'AKSHYA_TRITIYA', 'BUDDHA_PURNIMA', 'GANGA_DUSSEHRA', 'JAGANNATH_RATHYATRA', 'GURU_PURNIMA', 'NAG_PANCHAMI', 'RAKSHA_BANDHAN', 'MARRIAGE_DAY']


In [11]:
hist = df.filter((F.col(time_col) >= TRAIN_START) & (F.col(time_col) <= VAL_END))

# --- 3a target ---------------------------------------------------------------
r = hist.select(
    F.count(F.lit(1)).alias("N_ROWS"),
    F.sum(F.when(F.col(target_col).is_null(), 1).otherwise(0)).alias("N_NULL"),
    F.sum(F.when(F.col(target_col) < 0, 1).otherwise(0)).alias("N_NEG"),
    F.sum(F.when(F.col(target_col) != F.round(F.col(target_col)), 1).otherwise(0)).alias("N_NON_INTEGER"),
).collect()[0].as_dict()
print("3a target:", r)
assert r["N_NULL"] == 0 and r["N_NEG"] == 0, "Target has nulls or negatives -- fix the source table."
assert r["N_NON_INTEGER"] == 0, "Target has non-integer values; a count likelihood needs integers."

# --- 3b one row per day, no gaps --------------------------------------------
span = (hist.group_by(group_col)
            .agg(F.count(F.lit(1)).alias("N"), F.min(time_col).alias("D0"), F.max(time_col).alias("D1"))
            .with_column("EXPECTED", F.datediff("day", F.col("D0"), F.col("D1")) + 1))
r = span.select(
    F.count(F.lit(1)).alias("N_SERIES"),
    F.sum(F.when(F.col("N") < F.col("EXPECTED"), 1).otherwise(0)).alias("N_WITH_GAPS"),
    F.sum(F.when(F.col("N") > F.col("EXPECTED"), 1).otherwise(0)).alias("N_WITH_DUPLICATE_DAYS"),
    F.sum(F.when(F.col("D1") < F.lit(VAL_END), 1).otherwise(0)).alias("N_ENDING_BEFORE_VAL_END"),
).collect()[0].as_dict()
print("3b continuity:", r)
assert r["N_WITH_GAPS"] == 0 and r["N_WITH_DUPLICATE_DAYS"] == 0, "Series have missing or duplicate days."
if r["N_ENDING_BEFORE_VAL_END"]:
    print(f"   NOTE: {r['N_ENDING_BEFORE_VAL_END']} series stop before {VAL_END}; they cannot be forecast "
          f"from {FORECAST_START} and will be excluded in Step 2.")

# --- 3c / 3d static columns --------------------------------------------------
per_series = hist.group_by(group_col).agg(*[F.count_distinct(F.col(c)).alias(f"S{i}")
                                            for i, c in enumerate(static_covariates)])
mx = per_series.select(*[F.max(F.col(f"S{i}")).alias(f"S{i}")
                         for i in range(len(static_covariates))]).collect()[0].as_dict()
not_constant = [c for i, c in enumerate(static_covariates) if (mx[f"S{i}"] or 0) > 1]
nulls = hist.select(*[F.sum(F.when(F.col(c).is_null(), 1).otherwise(0)).alias(f"S{i}")
                      for i, c in enumerate(static_covariates)]).collect()[0].as_dict()
has_null = [c for i, c in enumerate(static_covariates) if (nulls[f"S{i}"] or 0) > 0]
print("3c static columns varying within a series:", not_constant or "none")
print("3d static columns with nulls             :", has_null or "none")
assert not not_constant, f"These static columns change within a series: {not_constant}"
assert not has_null, f"These static columns contain nulls: {has_null}"

# --- 3e festive columns national ---------------------------------------------
cal_src = df.filter((F.col(time_col) >= TRAIN_START) & (F.col(time_col) <= FORECAST_END))
per_date = cal_src.group_by(time_col).agg(*[F.count_distinct(F.col(c)).alias(f"C{i}")
                                            for i, c in enumerate(festive_covariates)])
mx = per_date.select(*[F.max(F.col(f"C{i}")).alias(f"C{i}")
                       for i in range(len(festive_covariates))]).collect()[0].as_dict()
regional = [c for i, c in enumerate(festive_covariates) if (mx[f"C{i}"] or 0) > 1]
print("3e festive columns that differ across series on the same date:", regional or "none")
assert not regional, (
    f"{regional} vary across series on the same date (regional festivals?). A single shared calendar "
    "would apply one region's dates to everyone. Move these to per-series covariates before continuing.")

3a target: {'N_ROWS': 40272756, 'N_NULL': 0, 'N_NEG': 0, 'N_NON_INTEGER': 0}
3b continuity: {'N_SERIES': 32244, 'N_WITH_GAPS': 0, 'N_WITH_DUPLICATE_DAYS': 0, 'N_ENDING_BEFORE_VAL_END': 0}
3c static columns varying within a series: none
3d static columns with nulls             : none
3e festive columns that differ across series on the same date: none


In [12]:
vocab = {}
for c in static_covariates:
    values = sorted(str(r[0]) for r in hist.select(F.col(c)).distinct().collect())
    vocab[c] = ["__UNKNOWN__"] + values
    print(f"  {c:20s} {len(values):5d} categories   e.g. {values[:3]}")

with open(vocab_path, "w") as f:
    json.dump(vocab, f, indent=1)
print(f"\nVocabulary written -> {vocab_path}")

  PARENT_DEALER_CODE    1102 categories   e.g. ['10001', '10003', '10007']
  MODEL_FAMILY            13 categories   e.g. ['DESTINI', 'GLAMOUR', 'HF 100']
  BRAKE_TYPE               2 categories   e.g. ['DISC', 'DRUM']
  IGNITION_TYPE            2 categories   e.g. ['KICK', 'SELF']
  WHEEL_TYPE               4 categories   e.g. ['CAST', 'CASTI', 'SHEET METAL']
  COLOUR                  27 categories   e.g. ['BLACK', 'BLACK AND ACCENT', 'BLACK GREY']
  X_CITY_CATEGORY          3 categories   e.g. ['RURAL', 'UNSPECIFIED', 'URBAN']
  ZONAL_OFFICE_NAME        5 categories   e.g. ['Zonal Office - Central', 'Zonal Office - East', 'Zonal Office - North']
  DEALER_CITY            826 categories   e.g. ['Aburoad', 'Achhalda', 'Adilabad']

Vocabulary written -> ./category_vocab.json


In [26]:
import shutil

shutil.rmtree(local_train_dir, ignore_errors=True)
os.makedirs(local_train_dir)

chunk_cols = [time_col, group_col, target_col] + static_covariates
dealers = sorted(r[0] for r in hist.select(dealer_col).distinct().collect())
chunks = [dealers[i:i + CHUNK_SIZE] for i in range(0, len(dealers), CHUNK_SIZE)]
print(f"{len(dealers)} dealers -> {len(chunks)} chunks")

for i, chunk in enumerate(chunks):
    part = hist.select(*chunk_cols).filter(F.col(dealer_col).isin(chunk)).to_pandas()
    part[target_col] = part[target_col].astype(np.float32)
    part.to_parquet(os.path.join(local_train_dir, f"chunk_{i:04d}.parquet"), index=False)
    print(f"  chunk {i+1}/{len(chunks)}: {len(part):,} rows, {part[group_col].nunique():,} series")
    del part
    gc.collect()

1102 dealers -> 12 chunks
  chunk 1/12: 3,866,904 rows, 3,096 series
  chunk 2/12: 4,435,199 rows, 3,551 series
  chunk 3/12: 4,473,918 rows, 3,582 series
  chunk 4/12: 4,021,780 rows, 3,220 series
  chunk 5/12: 4,411,468 rows, 3,532 series
  chunk 6/12: 3,771,980 rows, 3,020 series
  chunk 7/12: 3,861,908 rows, 3,092 series
  chunk 8/12: 3,994,302 rows, 3,198 series
  chunk 9/12: 3,594,622 rows, 2,878 series
  chunk 10/12: 2,978,865 rows, 2,385 series
  chunk 11/12: 858,063 rows, 687 series
  chunk 12/12: 3,747 rows, 3 series


In [27]:
cal = (cal_src.group_by(time_col)
              .agg(*[F.max(F.col(c)).alias(f"C{i}") for i, c in enumerate(festive_covariates)])
              .to_pandas())
cal.columns = [time_col] + festive_covariates          # restore real names (aliases avoid quoting issues)
cal[time_col] = pd.to_datetime(cal[time_col])
cal = cal.sort_values(time_col).reset_index(drop=True)

full = pd.date_range(TRAIN_START, FORECAST_END, freq=FREQ)
missing = full.difference(cal[time_col])
assert len(missing) == 0, (
    f"Calendar is missing {len(missing)} dates (first: {list(missing[:3])}). If the gap is at the end, "
    f"extend {TABLE_NAME} to {FORECAST_END}.")
assert cal[festive_covariates].notna().all().all(), "Festive columns contain nulls."

dt = cal[time_col].dt
cal["DOW_SIN"] = np.sin(2 * np.pi * dt.dayofweek / 7).astype(np.float32)
cal["DOW_COS"] = np.cos(2 * np.pi * dt.dayofweek / 7).astype(np.float32)
cal["IS_MONTH_END"]      = (dt.day >= dt.days_in_month - 2).astype(np.float32)
cal["IS_MONTH_START"]    = (dt.day <= 3).astype(np.float32)
cal["DAYS_TO_MONTH_END"] = ((dt.days_in_month - dt.day) / 31.0).astype(np.float32)
calendar_covariates = ["DOW_SIN", "DOW_COS", "IS_MONTH_END", "IS_MONTH_START", "DAYS_TO_MONTH_END"]

cal[festive_covariates] = cal[festive_covariates].astype(np.float32)
future_covariates = festive_covariates + calendar_covariates
cal[[time_col] + future_covariates].to_parquet(calendar_path, index=False)

fc = cal[(cal[time_col] >= FORECAST_START) & (cal[time_col] <= FORECAST_END)]
print(f"Calendar: {cal[time_col].min().date()} -> {cal[time_col].max().date()} ({len(cal)} days), "
      f"{len(future_covariates)} future covariates")
for anchor in ["N", "D"]:
    if anchor in fc.columns:
        print(f"  '{anchor}' falls on: {[d.date().isoformat() for d in fc.loc[fc[anchor] != 0, time_col]]}")

Calendar: 2023-04-01 -> 2026-12-10 (1350 days), 57 future covariates
  'N' falls on: ['2026-10-11']
  'D' falls on: ['2026-11-08']


In [28]:
roles = dict(
    time_col=time_col, group_col=group_col, dealer_col=dealer_col, target_col=target_col, freq=FREQ,
    static_covariates=static_covariates, festive_covariates=festive_covariates,
    calendar_covariates=calendar_covariates, future_covariates=future_covariates,
    train_start=TRAIN_START, train_end=TRAIN_END, val_start=VAL_START, val_end=VAL_END,
    forecast_start=FORECAST_START, forecast_end=FORECAST_END,
    input_chunk_length=INPUT_CHUNK_LENGTH, output_chunk_length=OUTPUT_CHUNK_LENGTH,
    vocab_path=vocab_path, calendar_path=calendar_path,
)
with open(roles_path, "w") as f:
    json.dump(roles, f, indent=2)
print(f"Roles written -> {roles_path}")

session.close()
print("Snowflake session closed.")

Roles written -> ./column_roles.json
Snowflake session closed.
